In [ ]:
!pip install --upgrade keras-nlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.2/465.2 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 64.9 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.0 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.0.5 which is incompatible.


In [ ]:
import keras

print(keras.__version__)

3.0.5


In [ ]:
!pip install opencc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.8/779.8 kB 12.2 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/chinese-poetry/chinese-poetry.git

Cloning into 'chinese-poetry'...
remote: Enumerating objects: 7309, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 7309 (delta 38), reused 77 (delta 20), pack-reused 7195
Receiving objects: 100% (7309/7309), 199.68 MiB | 13.67 MiB/s, done.
Resolving deltas: 100% (5326/5326), done.
Updating files: 100% (2285/2285), done.


In [ ]:
from keras_nlp.models import GPT2CausalLMPreprocessor
from keras_nlp.models import GPT2CausalLM
from keras.optimizers import schedules, Adam
from keras.losses import SparseCategoricalCrossentropy
import tensorflow as tf
import os, json
from opencc import OpenCC

keras.mixed_precision.set_global_policy("mixed_float16")

# 取得簡體中文版中文詩詞的全唐詩目錄下的唐詩, 這些是JSON檔案
poem_collection = []
for file in os.listdir("chinese-poetry/全唐诗"):
    if ".json" not in file or "poet" not in file:
        continue
    full_filename = "%s/%s" % ("chinese-poetry/全唐诗", file)
    with open(full_filename, "r") as f:
        content = json.load(f)
        poem_collection.extend(content)
# 將唐詩轉換成段落串列, 並且用OpenCC將簡體字轉換成繁體字
cc = OpenCC("s2tw.json")
paragraphs = [cc.convert("".join(data["paragraphs"]))
              for data in poem_collection]
# 顯示第1個段落的唐詩
print(paragraphs[0])

至日書雲，雲作何色。眼裡無筋，青黃赤白。


In [ ]:
# 使用TensorFlow的Dataset物件來批次載入唐詩的段落資料
train_ds = (
    tf.data.Dataset.from_tensor_slices(paragraphs)
    .batch(16)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)
# 因為唐詩的資料量並不小, 我們準備只取出500批次的樣本資料
train_ds = train_ds.take(500)
# 建立GPT-2預處理, 為了加速將文字長度改成128（預設值是1024）
preprocessor = GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en",
    sequence_length=128,
)
# 載入GPT-2模型和指定預處理
gpt2_lm = GPT2CausalLM.from_preset(
    "gpt2_base_en",
    preprocessor=preprocessor
)
num_epochs = 2   # 因為只是展示, 所以訓練週期只有2次
# 使用學習率指數衰減物件來建立自訂參數的Adam優化器
learning_rate = schedules.PolynomialDecay(
    5e-4,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)
# 編譯模型
gpt2_lm.compile(
    optimizer=Adam(learning_rate),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=["accuracy"],
)
# 訓練模型
gpt2_lm.fit(train_ds, epochs=num_epochs)
# 測試使用GPT-2來生成唐詩
output = gpt2_lm.generate("昨夜雨疏風驟", max_length=200)
print(output)

100%|██████████| 448/448 [00:00<00:00, 1.27MB/s]
100%|██████████| 446k/446k [00:01<00:00, 443kB/s]
100%|██████████| 0.99M/0.99M [00:01<00:00, 792kB/s]
100%|██████████| 484/484 [00:00<00:00, 332kB/s]
100%|██████████| 475M/475M [00:31<00:00, 16.0MB/s]


Epoch 1/2
500/500 ━━━━━━━━━━━━━━━━━━━━ 258s 248ms/step - accuracy: 0.2676 - loss: 2.4311
Epoch 2/2
500/500 ━━━━━━━━━━━━━━━━━━━━ 163s 249ms/step - accuracy: 0.3285 - loss: 2.0774
昨夜雨疏風驟消，百下花陰晝色靜。苔苔塵坐城靜，清陰自絕霜霜霞。
